# Global Model Synthesis

This notebook implements a **global synthesis model** that integrates features and diagnostics generated by the rolling surrogate regime analysis into a single, stable analytical view.

While the rolling surrogate workflow is designed to detect and characterize local regime shifts, this global model is designed to:
- summarize system behavior across all observed regimes,
- extract persistent, cross-regime structure,
- and provide a consistent reference frame against which local deviations can be interpreted.

The two notebooks are therefore complementary:
- the rolling surrogate analysis focuses on *change, drift, and regime structure*,
- the global model focuses on *aggregation, comparison, and long-horizon structure*.

---

## Conceptual Role

The global model answers a different class of questions than the rolling surrogate:

- *Which features matter consistently across regimes?*
- *Which signals explain long-term variation rather than transient fluctuations?*
- *How do different regimes relate to one another in a lower-dimensional representation?*
- *Which combinations of features define typical, atypical, or extreme system states?*

Rather than treating the system as a sequence of local behaviors, the global model treats it as a **population of states** and attempts to characterize the geometry and structure of that population.

---

## Analytical Structure

The workflow proceeds in four stages:

### 1. Feature Integration
The per-run feature table generated by the rolling analysis is loaded and cleaned.

This includes:
- selecting relevant features,
- handling missing values,
- normalizing or standardizing where appropriate,
- and optionally engineering composite or interaction features.

The result is a consistent, analysis-ready feature matrix spanning all runs.

---

### 2. Dimensionality Reduction and Structure Discovery
High-dimensional feature spaces are projected into lower-dimensional representations (e.g., via PCA or related methods) to reveal dominant axes of variation.

This supports:
- visualization of system state distributions,
- identification of clusters, gradients, or outliers,
- and reasoning about relationships between regimes.

These projections are descriptive tools rather than predictive models.

---

### 3. Global Predictive Modeling
A single global model is fit across all runs to capture long-horizon explanatory structure.

Unlike the rolling surrogate models, this model is:
- trained on the full dataset,
- optimized for stability rather than sensitivity,
- and used to understand persistent drivers rather than regime transitions.

Feature importance, partial dependence, or similar diagnostics are used to interpret global relationships.

---

### 4. Regime Comparison and Contextualization
Outputs from the rolling analysis (regime labels, instability flags, or local driver changes) are merged with the global representations.

This allows:
- comparison of regimes in the same latent space,
- identification of which regimes are structurally similar or distinct,
- and interpretation of local changes in the context of global structure.

---

## Design Principles

- **Complementarity:** The global model does not replace the rolling analysis; it contextualizes it.
- **Stability over sensitivity:** The global model prioritizes robustness and interpretability over responsiveness to short-term change.
- **Geometry over sequence:** The system is analyzed as a distribution of states, not primarily as a time series.
- **Interpretability over optimization:** Emphasis is placed on understanding structure rather than maximizing predictive accuracy.

---

## Summary

The global model provides a long-horizon, integrative view of system behavior that complements the regime-aware rolling surrogate analysis.

Together, the two workflows support a multi-scale understanding of complex, evolving systems:
- the rolling analysis detects and explains *when and how the system changes*,
- the global model explains *what the system is, in aggregate*.

This dual perspective enables both operational awareness and structural understanding, without requiring strong mechanistic assumptions or fixed regime definitions.

In [ ]:
# Dependency verification and imports for the global model pipeline.
import sys, subprocess, importlib.util, warnings

REQUIRED_PACKAGES = {
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "matplotlib": "matplotlib>=3.7",
    "scipy": "scipy>=1.11",
    "sklearn": "scikit-learn>=1.3",
    "shap": "shap>=0.44",
    "lime": "lime>=0.2.0.1",
}

for import_name, package_spec in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_spec}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_spec])

from pathlib import Path
import json

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau,
    mannwhitneyu,
)

from sklearn.preprocessing import (
    StandardScaler,
    PowerTransformer,
    QuantileTransformer,
    OneHotEncoder,
)
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
)

from sklearn.linear_model import (
    ElasticNetCV,
    LogisticRegressionCV,
)

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    IsolationForest,
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    roc_auc_score,
    balanced_accuracy_score,
)

from sklearn.covariance import MinCovDet

import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore", category=FutureWarning)
print("Dependency check complete.")

In [ ]:
# Inputs and shared schema.
FEATURE_DIR = Path("pipeline_outputs/features")
FEATURE_FILE = "feature_table.csv"
FEATURE_TABLE_PATH = FEATURE_DIR / FEATURE_FILE

GLOBAL_OUTPUT_DIR = FEATURE_DIR.parent / "global_model_outputs"
GLOBAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = FEATURE_DIR / "pipeline_manifest.json"
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8") as handle:
        PIPELINE_MANIFEST = json.load(handle)
else:
    PIPELINE_MANIFEST = {}

RUN_ID_COL = PIPELINE_MANIFEST.get("run_id_column", "run_id")
DATE_COL = PIPELINE_MANIFEST.get("date_column", "run_start_date")
TARGET_COL = PIPELINE_MANIFEST.get("target_column", "target_metric_primary")

USER_START_DATE = None
earliest_date = pd.to_datetime(USER_START_DATE, errors="coerce")

MANUAL_RUN_METADATA_CSV = Path("pipeline_inputs/manual_run_metrics.csv")
manual_cols_to_add = [
    "feature_20",
    "feature_4_1_dwell_median",
    "feature_4_1_dwell_delta",
]
prefix_manual_cols = "manual__"

df = pd.read_csv(FEATURE_TABLE_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

if pd.notna(earliest_date):
    df = df.loc[df[DATE_COL] >= earliest_date].copy()

if RUN_ID_COL not in df.columns:
    raise ValueError(f"Feature table must contain '{RUN_ID_COL}'.")

df[RUN_ID_COL] = df[RUN_ID_COL].astype(str).str.strip()

if MANUAL_RUN_METADATA_CSV.exists():
    manual_df = pd.read_csv(MANUAL_RUN_METADATA_CSV)
    if RUN_ID_COL not in manual_df.columns:
        raise ValueError(f"Manual metrics file must contain '{RUN_ID_COL}'.")

    manual_df[RUN_ID_COL] = manual_df[RUN_ID_COL].astype(str).str.strip()

    if manual_cols_to_add == ["ALL"]:
        cols = [col for col in manual_df.columns if col != RUN_ID_COL]
    else:
        missing = [col for col in manual_cols_to_add if col not in manual_df.columns]
        if missing:
            raise ValueError(f"Requested manual columns not found: {missing}")
        cols = manual_cols_to_add

    manual_sel = manual_df[[RUN_ID_COL] + cols].drop_duplicates(RUN_ID_COL, keep="first").set_index(RUN_ID_COL)
    if prefix_manual_cols:
        manual_sel = manual_sel.rename(columns={col: f"{prefix_manual_cols}{col}" for col in cols})
        added_cols = [f"{prefix_manual_cols}{col}" for col in cols]
    else:
        added_cols = cols

    df = df.join(manual_sel, on=RUN_ID_COL)
    n_filled = df[added_cols].notna().any(axis=1).sum() if added_cols else 0
    print(f"Joined manual metrics: added {len(added_cols)} columns; {n_filled}/{len(df)} rows filled.")
else:
    print(f"Manual metrics file not found at {MANUAL_RUN_METADATA_CSV}; continuing without it.")

missing_tokens = {"nan": np.nan, "": np.nan, "NA": np.nan, "N/A": np.nan, "null": np.nan, "None": np.nan}

for col in df.columns:
    if df[col].dtype == "object":
        series = df[col].astype(str)
        if series.str.contains("%", na=False).any():
            df[col] = series.str.strip().str.replace("%", "", regex=False).replace(missing_tokens)
            df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print(f"Final shape: {df.shape}")
display(df.head())

In [ ]:
# ---------------------------
# 0) Outputs folder (same directory as original data)
# ---------------------------
out_dir = GLOBAL_OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------
# 1) Select PCA columns (any column containing keys anywhere)
# ---------------------------
pca_keys = [
    "feature_1_metric_01",
    "feature_4_1_mean",
    "feature_5",
    "feature_6",
    "feature_1_block",
    "feature_3",
    "feature_7",
]
pca_cols = [c for c in df.columns if any(k in c for k in pca_keys)]
print(f"Selected {len(pca_cols)} columns for PCA.")
if len(pca_cols) == 0:
    raise ValueError("No PCA columns matched. Check spelling/case in your keys vs column names.")

# ---------------------------
# 2) Build PCA matrix (numeric + clean)
# ---------------------------
X_raw = df[pca_cols].copy()
X_raw = X_raw.apply(pd.to_numeric, errors="coerce")
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

# Drop constant columns (keeps PCA/transform stable)
var0 = X_raw.var(axis=0, ddof=0)
const_cols = var0[var0 == 0].index.tolist()
if const_cols:
    print(f"Dropping {len(const_cols)} constant columns (examples): {const_cols[:5]}")
    X_raw = X_raw.drop(columns=const_cols)

print("Matrix for transform:", X_raw.shape)

# ---------------------------
# 3) Column-by-column transform with fallback (silence warnings ONLY here)
#    - Yeo–Johnson where possible
#    - Quantile->Normal fallback where YJ fails
# ---------------------------
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    yj = PowerTransformer(method="yeo-johnson", standardize=False)
    qt = QuantileTransformer(
        output_distribution="normal",
        n_quantiles=min(100, X_raw.shape[0]),
        random_state=0
    )

    X_tx = pd.DataFrame(index=X_raw.index)
    failed_cols = []

    for col in X_raw.columns:
        xcol = X_raw[[col]].values
        try:
            X_tx[col] = yj.fit_transform(xcol).ravel()
        except Exception:
            failed_cols.append(col)
            X_tx[col] = qt.fit_transform(xcol).ravel()

print(f"Yeo–Johnson failed on {len(failed_cols)} columns.")
if failed_cols:
    print("Failed columns (first 20):")
    for c in failed_cols[:20]:
        print("  ", c)

# ---------------------------
# 4) Standardize + PCA
# ---------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_tx)

pca = PCA()
scores = pca.fit_transform(X_scaled)

pc_names = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
score_df = pd.DataFrame(scores, columns=pc_names, index=X_tx.index)

explained_ratio = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained_ratio)
pcs = np.arange(1, len(explained_ratio) + 1)

# Export PCA scores (optionally with run identifiers if present)
id_cols = [c for c in ["run_id", "run_start_date", "unit_label", "group_label"] if c in df.columns]
pca_scores_out = pd.concat([df[id_cols].loc[score_df.index], score_df], axis=1) if id_cols else score_df

pca_scores_out.to_csv(out_dir / "pca_scores.csv", index=False)

# ---------------------------
# 5) Scree plots
# ---------------------------
plt.figure(figsize=(10, 5))
plt.plot(pcs, explained_ratio, marker="o")
plt.xticks(pcs)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot (YJ with Quantile fallback)")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(pcs, cum_explained, marker="o")
plt.xticks(pcs)
plt.xlabel("Principal Component")
plt.ylabel("Cumulative Explained Variance")
plt.ylim(0, 1.05)
plt.title("Cumulative Explained Variance (YJ with Quantile fallback)")
plt.grid(True, alpha=0.3)
plt.show()

# ============================================================
# EXPORTS FOR INTERPRETABILITY
# ============================================================

# ---------------------------
# 6) Square cosine (cos²) for variables vs PCs
#    For standardized PCA, corr(feature_j, PC_k) = loading_jk * sqrt(eigenvalue_k)
#    cos² = corr²
# ---------------------------
feature_names = list(X_tx.columns)
components = pca.components_                      # shape: (n_pc, n_features)
eigenvalues = pca.explained_variance_             # length: n_pc

# Correlation matrix (features x PCs)
corr_feat_pc = (components.T * np.sqrt(eigenvalues)).astype(float)  # (n_features, n_pc)
corr_feat_pc_df = pd.DataFrame(corr_feat_pc, index=feature_names, columns=pc_names)

# Square cosine matrix
cos2_df = corr_feat_pc_df ** 2

# Export full matrices (wide)
corr_feat_pc_df.to_csv(out_dir / "feature_vs_pc_correlation_from_loadings.csv")
cos2_df.to_csv(out_dir / "feature_vs_pc_cos2.csv")

# Export “top contributors” per PC (long)
top_n = min(20, cos2_df.shape[0])  # export top 20 per PC by default
top_cos2_rows = []
for pc in pc_names:
    s = cos2_df[pc].sort_values(ascending=False)
    top = s.head(top_n)
    for rank, (feat, val) in enumerate(top.items(), start=1):
        top_cos2_rows.append({"PC": pc, "Rank": rank, "Feature": feat, "cos2": float(val)})

top_cos2_df = pd.DataFrame(top_cos2_rows)
top_cos2_df.to_csv(out_dir / "top_feature_contributors_by_pc_cos2.csv", index=False)

# ---------------------------
# 7) Pearson correlation test between ORIGINAL FEATURES (X_raw) and PC scores
#    Export r and p-value for each feature-PC pair (long)
# ---------------------------
corr_test_rows = []
for feat in X_raw.columns:
    x = X_raw[feat].values
    for pc in pc_names:
        y = score_df[pc].values
        # pearsonr returns (r, p); handles constant inputs by raising ValueError
        try:
            r, p = pearsonr(x, y)
        except Exception:
            r, p = np.nan, np.nan
        corr_test_rows.append({
            "Feature": feat,
            "PC": pc,
            "r": r,
            "p_value": p
        })

corr_test_df = pd.DataFrame(corr_test_rows)
corr_test_df.to_csv(out_dir / "pearson_feature_vs_pc_r_and_p.csv", index=False)

print(f"\nExports written to:\n  {out_dir}")
print("Files:")
print("  - feature_vs_pc_cos2.csv")
print("  - top_feature_contributors_by_pc_cos2.csv")
print("  - feature_vs_pc_correlation_from_loadings.csv")
print("  - pearson_feature_vs_pc_r_and_p.csv")

In [ ]:
# ------------------------------------------------------------
# Assemble modeling dataset:
#   (ALL original features EXCEPT PCA-source features EXCEPT user-excluded) + (PCs) -> target
# ------------------------------------------------------------

# ===== USER CONTROLS (edit these) =====
# Select which principal components to include in training data
pcs_to_include = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7",
                 "PC8", "PC9", "PC10", "PC11", "PC12", "PC13", 
                 "PC14", "PC15", "PC16"]  
# Select any features for exclusion
exclude_exact = [
    "run_id",
    "run_start_date", "feature_1_setpoint", 
    "feature_8_mean_WINDOW_1", "feature_8_mean_WINDOW_3", 
    "feature_9_lag1_autocorr_WINDOW_1", 
    "feature_9_lag1_autocorr_WINDOW_2", "feature_9_lag1_autocorr_WINDOW_3", 
    "feature_9_dominant_freq_WINDOW_1", "feature_9_dominant_freq_WINDOW_2", 
    "feature_9_dominant_freq_WINDOW_3", "feature_9_resid_std_WINDOW_1", 
    "feature_9_resid_std_WINDOW_2", "feature_9_resid_std_WINDOW_3"
]
exclude_contains = [
    # Add generic substrings to exclude if needed.
]
# =====================================

# Define target
target_col = TARGET_COL
if target_col not in df.columns:
    raise ValueError(f"Target column not found: {target_col}")

pca_source_cols = [c for c in df.columns if any(k in c for k in pca_keys)]
print(f"Excluding {len(pca_source_cols)} PCA-source features from original data.")

# Build exclusion set
exclude_cols = set(pca_source_cols + [target_col])

# Add exact exclusions (if present)
exclude_cols.update([c for c in exclude_exact if c in df.columns])

# Add "contains" exclusions
for c in df.columns:
    if any(substr in c for substr in exclude_contains):
        exclude_cols.add(c)

print(f"Total excluded columns (PCA + target + user exclusions): {len(exclude_cols)}")

# Build list of ORIGINAL features to retain
retain_original_cols = [c for c in df.columns if c not in exclude_cols]
print(f"Retaining {len(retain_original_cols)} original features after exclusions.")

# Choose PCs to include (validate they exist)
pc_cols = [pc for pc in pcs_to_include if pc in score_df.columns]
missing_pcs = [pc for pc in pcs_to_include if pc not in score_df.columns]
if missing_pcs:
    raise ValueError(f"These PCs are not present in score_df: {missing_pcs}")
print(f"Including PCs: {pc_cols}")

# Assemble modeling dataframe
model_df = pd.concat(
    [
        df[retain_original_cols].copy(),
        score_df[pc_cols].copy(),
        df[[target_col]].copy(),
    ],
    axis=1
)

# Drop rows with missing target
before = model_df.shape[0]
model_df = model_df.dropna(subset=[target_col]).copy()
after = model_df.shape[0]

print(f"Dropped {before - after} rows with missing target.")
print(f"Final modeling dataset shape: {model_df.shape}")

# Ensure numeric target
model_df[target_col] = pd.to_numeric(model_df[target_col], errors="coerce")

# Quick inspection
display(model_df.head())

# OPTIONAL: see exactly what was excluded (helpful for debugging)
# excluded_list = sorted(list(exclude_cols))
# display(excluded_list[:50])
# print(f"... total excluded: {len(excluded_list)}")

from pathlib import Path

# ---------------------------
# Export prepared training data (model_df) to global_model_outputs
# ---------------------------
out_dir = GLOBAL_OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

model_df.to_csv(out_dir / "training_data_prepared.csv", index=False)

print(f"\nTraining data exported to:\n  {out_dir / 'training_data_prepared.csv'}")

In [ ]:
# ============================================================
# Feature engineering (run after model_df assembly, before FS/training)
#   Goal (this block): add ONE engineered feature DIRECTLY to model_df:
#     feat__z_f8_yj_x_z_f9__WINDOW_2
#       = z( YJ(feature_8_mean_WINDOW_2) ) * z( feature_9_median_WINDOW_2 )
# -------------------------------------------------------------------

# ---------------------------
# 0) Assumptions / optional reload
# ---------------------------
# Assumes `model_df` exists in memory from the previous block.
# If you prefer to reload from disk, uncomment:
# out_dir = GLOBAL_OUTPUT_DIR
# model_df = pd.read_csv(out_dir / "training_data_prepared.csv")

pt_col   = "feature_8_mean_WINDOW_2"
secondary_col = "feature_9_median_WINDOW_2"

# Engineered feature name
interaction_col = "feat__z_f8_yj_x_z_f9__WINDOW_2"

# ---------------------------
# 1) Guardrails: required columns exist
# ---------------------------
missing = [c for c in [pt_col, secondary_col] if c not in model_df.columns]
if missing:
    raise ValueError(
        "Missing required columns in model_df:\n"
        f"  {missing}\n"
        "Confirm these columns are retained in the assembly block (retain_original_cols / exclusions)."
    )

# Ensure numeric (in-place on model_df)
model_df[pt_col] = pd.to_numeric(model_df[pt_col], errors="coerce")
model_df[secondary_col] = pd.to_numeric(model_df[secondary_col], errors="coerce")

# Minimal imputation just for these two columns (consistent with upstream median impute)
for c in [pt_col, secondary_col]:
    if model_df[c].isna().any():
        model_df[c] = model_df[c].fillna(model_df[c].median())

# ---------------------------
# 2) Build the engineered interaction feature: z(YJ(pt)) × z(secondary signal)
# ---------------------------

# Yeo–Johnson transform pt (variance stabilization; works with non-positive values)
yj = PowerTransformer(method="yeo-johnson", standardize=False)
pt_yj = yj.fit_transform(model_df[[pt_col]].values).ravel()

def zscore(x: np.ndarray) -> np.ndarray:
    """Return z-scored array using population std (ddof=0). Raises if sd == 0."""
    x = np.asarray(x, dtype=float)
    sd = x.std(ddof=0)
    if sd == 0 or not np.isfinite(sd):
        raise ValueError("Cannot z-score: standard deviation is 0 or non-finite.")
    return (x - x.mean()) / sd

pt_z = zscore(pt_yj)
secondary_z = zscore(model_df[secondary_col].values)

# Add/overwrite engineered feature directly on model_df
model_df[interaction_col] = pt_z * secondary_z

# QC: ensure finite values
n_bad = (~np.isfinite(model_df[interaction_col].values)).sum()
if n_bad:
    raise ValueError(f"{interaction_col} contains {n_bad} non-finite values; check inputs for inf/NaN.")

# ---------------------------
# 3) Quick summary + preview
# ---------------------------
print("Engineered feature added to model_df:")
print(f"  - {interaction_col} = z(YJ({pt_col})) * z({secondary_col})")
print(f"  - mean={model_df[interaction_col].mean():.3f}, std={model_df[interaction_col].std(ddof=0):.3f}, "
      f"min={model_df[interaction_col].min():.3f}, max={model_df[interaction_col].max():.3f}")

display(model_df[[pt_col, secondary_col, interaction_col]].head())

# ---------------------------
# 4) Export engineered dataset artifact (same filename as before)
# ---------------------------
out_dir = GLOBAL_OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

engineered_out = out_dir / "training_data_engineered.csv"
model_df.to_csv(engineered_out, index=False)

print(f"\nEngineered training data exported to:\n  {engineered_out}")

# Downstream feature selection / training should use:
#   model_df (in-memory) OR engineered_out (on disk)

In [ ]:
# Feature engineering for optional run-level treatment metrics.
# Adds two engineered features directly to model_df:
#   1) feat__delta_f16_drop = min(feature_17 - feature_16, 0)
#   2) feat__z_f17_x_z_delta_f16_drop = z(feature_17) * z(feat__delta_f16_drop)

feature_16_col = "feature_16"
feature_17_col = "feature_17"

feat_drop = "feat__delta_f16_drop"
feat_inter = "feat__z_f17_x_z_delta_f16_drop"

missing = [col for col in [feature_16_col, feature_17_col] if col not in model_df.columns]
if missing:
    raise ValueError(
        "Missing required treatment metric columns in model_df:\n"
        f"  {missing}\n"
        "Confirm these columns are retained in the training data assembly block."
    )

model_df[feature_16_col] = pd.to_numeric(model_df[feature_16_col], errors="coerce")
model_df[feature_17_col] = pd.to_numeric(model_df[feature_17_col], errors="coerce")

for col in [feature_16_col, feature_17_col]:
    if model_df[col].isna().any():
        model_df[col] = model_df[col].fillna(model_df[col].median())

delta_treatment_metric = model_df[feature_17_col] - model_df[feature_16_col]
model_df[feat_drop] = np.minimum(delta_treatment_metric.values, 0.0)

def zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    sd = x.std(ddof=0)
    if sd == 0 or not np.isfinite(sd):
        raise ValueError("Cannot z-score: standard deviation is 0 or non-finite.")
    return (x - x.mean()) / sd

feature_17_z = zscore(model_df[feature_17_col].values)
drop_z = zscore(model_df[feat_drop].values)
model_df[feat_inter] = feature_17_z * drop_z

for col in [feat_drop, feat_inter]:
    n_bad = (~np.isfinite(model_df[col].values)).sum()
    if n_bad:
        raise ValueError(f"{col} contains {n_bad} non-finite values; check input metrics.")

print("\nEngineered treatment metric features added to model_df:")
for col in [feat_drop, feat_inter]:
    print(
        f"  - {col}: mean={model_df[col].mean():.3f}, std={model_df[col].std(ddof=0):.3f}, "
        f"min={model_df[col].min():.3f}, max={model_df[col].max():.3f}"
    )

display(model_df[[feature_16_col, feature_17_col, feat_drop, feat_inter]].head())

engineered_out = GLOBAL_OUTPUT_DIR / "training_data_engineered.csv"
GLOBAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_df.to_csv(engineered_out, index=False)

print(f"\nEngineered training data updated at:\n  {engineered_out}")
print(f"Final shape: {model_df.shape}")

In [ ]:
# ------------------------------------------------------------
# Elastic Net feature selection (stable for wide/shallow)
#   - Handles correlated predictors
#   - Produces a sparse (selected) feature set via non-zero coefficients
# ------------------------------------------------------------

# Target column (uses your existing variable)
target_col = TARGET_COL

# Split X/y
X = model_df.drop(columns=[target_col]).copy()
y = pd.to_numeric(model_df[target_col], errors="coerce").copy()

# (Safety) drop rows where target became NaN after coercion
mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

# Identify feature types
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print(f"Rows: {X.shape[0]} | Predictors: {X.shape[1]} (numeric={len(num_cols)}, categorical={len(cat_cols)})")

# Preprocess: scale numeric, one-hot encode categorical
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# ElasticNetCV:
# - l1_ratio controls mix of L1 (lasso) and L2 (ridge)
# - alphas searched automatically; you can widen/narrow this grid if needed
cv = KFold(n_splits=min(5, len(X)), shuffle=True, random_state=0)

enet = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    alphas=np.logspace(-4, 1, 60),
    cv=cv,
    max_iter=20000,
    random_state=0
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("enet", enet)
])

pipe.fit(X, y)

# Pull feature names after preprocessing (includes one-hot expanded columns)
feature_names = pipe.named_steps["prep"].get_feature_names_out()
coefs = pipe.named_steps["enet"].coef_

coef_s = pd.Series(coefs, index=feature_names).sort_values(key=np.abs, ascending=False)

# Selected features = non-zero coefficients (use a small tolerance for numerical noise)
tol = 1e-8
selected = coef_s[coef_s.abs() > tol]

print("\nElastic Net chosen hyperparameters:")
print(f"  alpha:    {pipe.named_steps['enet'].alpha_}")
print(f"  l1_ratio: {pipe.named_steps['enet'].l1_ratio_}")

print(f"\nSelected features (non-zero): {selected.shape[0]} / {len(feature_names)}")
display(selected.head(30).to_frame("coef"))

# Optional: keep lists for downstream modeling
selected_feature_names = selected.index.tolist()

# If you want a compact dataframe for modeling directly (post-preprocess), build it:
X_design = pd.DataFrame(pipe.named_steps["prep"].transform(X), columns=feature_names, index=X.index)
X_selected = X_design[selected_feature_names].copy()

print("\nShapes for downstream modeling:")
print("  X_design:", X_design.shape)
print("  X_selected:", X_selected.shape)

In [ ]:
# Random Forest regression with feature selection refit inside each CV fold.
target_col = TARGET_COL

X_full = model_df.drop(columns=[target_col]).copy()
y_full = pd.to_numeric(model_df[target_col], errors="coerce").copy()

mask = y_full.notna()
X_full, y_full = X_full.loc[mask], y_full.loc[mask]

def fit_regression_selector(X_train, y_train, random_state=0):
    """Fit preprocessing and Elastic Net selection on training rows only."""
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [col for col in X_train.columns if col not in num_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    cv_inner = KFold(n_splits=min(5, len(X_train)), shuffle=True, random_state=random_state)
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
        alphas=np.logspace(-4, 1, 60),
        cv=cv_inner,
        max_iter=20000,
        random_state=random_state,
    )

    pipe = Pipeline(steps=[("prep", preprocess), ("enet", enet)])
    pipe.fit(X_train, y_train)

    feature_names = pipe.named_steps["prep"].get_feature_names_out()
    coefs = pd.Series(pipe.named_steps["enet"].coef_, index=feature_names)
    selected_names = coefs.index[coefs.abs() > 1e-8].tolist()
    if not selected_names:
        selected_names = feature_names.tolist()

    return pipe, selected_names

def transform_selected(selector_pipe, X_data, selected_names):
    feature_names = selector_pipe.named_steps["prep"].get_feature_names_out()
    design = pd.DataFrame(
        selector_pipe.named_steps["prep"].transform(X_data),
        columns=feature_names,
        index=X_data.index,
    )
    return design[selected_names].copy()

n_splits = 5
n_repeats = 20

r2_scores = []
rmse_scores = []
selected_counts = []

for seed in range(n_repeats):
    cv_outer = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for train_idx, test_idx in cv_outer.split(X_full):
        X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
        y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]

        selector_pipe, selected_names = fit_regression_selector(X_train, y_train, random_state=seed)
        X_train_sel = transform_selected(selector_pipe, X_train, selected_names)
        X_test_sel = transform_selected(selector_pipe, X_test, selected_names)
        selected_counts.append(len(selected_names))

        rf = RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=seed,
        )

        rf.fit(X_train_sel, y_train)
        y_pred = rf.predict(X_test_sel)

        r2_scores.append(r2_score(y_test, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))

r2_scores = np.array(r2_scores)
rmse_scores = np.array(rmse_scores)

print("Random Forest regression performance with fold-local feature selection:")
print(f"R²:   mean={r2_scores.mean():.3f}, std={r2_scores.std():.3f}")
print(f"RMSE: mean={rmse_scores.mean():.3f}, std={rmse_scores.std():.3f}")
print("R² quantiles:", np.quantile(r2_scores, [0.1, 0.25, 0.5, 0.75, 0.9]))
print(f"Selected feature count: median={np.median(selected_counts):.0f}, range=({np.min(selected_counts)}, {np.max(selected_counts)})")

In [ ]:
rf_final = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=0
)

rf_final.fit(X, y)

In [ ]:
# --- SHAP for your final Random Forest model (regression) ---
# Assumes:
#   rf_final   -> trained RandomForestRegressor
#   X_selected -> DataFrame of features used for training

X_explain = X_selected.copy()

explainer = shap.Explainer(rf_final, X_explain)
shap_values = explainer(X_explain)

# Make matplotlib a bit more forgiving about label space
plt.rcParams["figure.autolayout"] = True

# --- Global importance (beeswarm) ---
shap.summary_plot(
    shap_values,
    X_explain,
    max_display=20,          # reduce if labels are long
    plot_size=(14, 6),       # <-- THIS is the key control
    show=True
)

# --- Global importance (bar) ---
shap.summary_plot(
    shap_values,
    X_explain,
    plot_type="bar",
    max_display=20,
    plot_size=(14, 6),
    show=True
)

In [ ]:
# ------------------------------------------------------------
# Anomaly scoring (VALID use-cases):
#   1) Input-space anomaly in PCA space (model-free): Mahalanobis + IsolationForest
#   2) Model-disagreement anomaly (diagnostic only): abs residual z-score
#
# Assumes you already have:
#   - df         : your filtered dataset (rows = runs)
#   - score_df   : PCA scores DataFrame with columns PC1..PCn (same index as df)
#   - model_df   : your prepared training data (optional for joining)
#   - rf_final   : trained regression model on X_selected (optional; used for residual anomaly)
#   - X_selected : feature matrix used to train rf_final (optional; aligned index)
#   - target_col : string target name (set below)
# ------------------------------------------------------------

# ===== USER CONTROLS =====
target_col = TARGET_COL
pcs_for_anomaly = [f"PC{i}" for i in range(1, 17)]   # choose PCs used for anomaly scoring (must exist in score_df)
iso_contamination = "auto"                           # or e.g. 0.1 if you want a fixed assumed fraction
random_state = 0
# =========================

# --- Validate and assemble PC matrix ---
pcs_for_anomaly = [pc for pc in pcs_for_anomaly if pc in score_df.columns]
if len(pcs_for_anomaly) == 0:
    raise ValueError("No PCs found for anomaly scoring. Check pcs_for_anomaly vs score_df columns.")

S = score_df[pcs_for_anomaly].copy()
S = S.replace([np.inf, -np.inf], np.nan)
S = S.fillna(S.median(numeric_only=True))

# ------------------------------------------------------------
# 1) Input-space anomaly score: Robust Mahalanobis distance (MinCovDet)
# ------------------------------------------------------------
mcd = MinCovDet(random_state=random_state).fit(S.values)
md2 = mcd.mahalanobis(S.values)          # squared robust Mahalanobis distance
md = np.sqrt(md2)

# Normalize to z-score for easier thresholding
md_z = (md - md.mean()) / (md.std(ddof=0) + 1e-12)

# ------------------------------------------------------------
# 2) Input-space anomaly score: Isolation Forest in PC space
# ------------------------------------------------------------
iso = IsolationForest(
    n_estimators=500,
    contamination=iso_contamination,
    random_state=random_state
).fit(S.values)

# sklearn: higher decision_function = more normal; invert so higher = more anomalous
iso_anom = -iso.decision_function(S.values)

# Normalize to z-score
iso_z = (iso_anom - iso_anom.mean()) / (iso_anom.std(ddof=0) + 1e-12)

# ------------------------------------------------------------
# 3) Model-disagreement anomaly (diagnostic only): absolute residual z-score
#    (Optional; only computed if rf_final and X_selected are available)
# ------------------------------------------------------------
resid_abs = None
resid_abs_z = None

if "rf_final" in globals() and "X_selected" in globals():
    # Ensure alignment
    common_idx = S.index.intersection(X_selected.index)
    if len(common_idx) >= 5 and target_col in df.columns:
        y_true = pd.to_numeric(df.loc[common_idx, target_col], errors="coerce")
        Xp = X_selected.loc[common_idx]
        y_pred = pd.Series(rf_final.predict(Xp), index=common_idx)

        resid = (y_true - y_pred)
        resid_abs = resid.abs()

        # Robust-ish z using MAD (more stable with small N)
        med = np.nanmedian(resid_abs)
        mad = np.nanmedian(np.abs(resid_abs - med)) + 1e-12
        resid_abs_z = (resid_abs - med) / (1.4826 * mad)

# ------------------------------------------------------------
# Assemble anomaly table
# ------------------------------------------------------------
id_cols = [c for c in ["run_id", "run_start_date", "unit_label", "group_label"] if c in df.columns]

anom_df = pd.DataFrame(index=S.index)
if id_cols:
    anom_df = pd.concat([df.loc[S.index, id_cols], anom_df], axis=1)

anom_df["mdistance"] = md
anom_df["mdistance_z"] = md_z
anom_df["iso_anomaly"] = iso_anom
anom_df["iso_anomaly_z"] = iso_z

# Combined input anomaly score (simple average of z-scores)
anom_df["input_anomaly_score_z"] = (anom_df["mdistance_z"] + anom_df["iso_anomaly_z"]) / 2.0

# Optional residual anomaly columns
if resid_abs is not None:
    anom_df.loc[resid_abs.index, "y_true"] = pd.to_numeric(df.loc[resid_abs.index, target_col], errors="coerce")
    anom_df.loc[resid_abs.index, "y_pred"] = y_pred
    anom_df.loc[resid_abs.index, "abs_residual"] = resid_abs
    anom_df.loc[resid_abs.index, "abs_residual_z"] = resid_abs_z

    # Combined diagnostic score: input anomaly + residual anomaly (diagnostic only)
    anom_df["diagnostic_anomaly_score_z"] = anom_df["input_anomaly_score_z"]
    anom_df.loc[resid_abs.index, "diagnostic_anomaly_score_z"] = (
        anom_df.loc[resid_abs.index, "input_anomaly_score_z"] + anom_df.loc[resid_abs.index, "abs_residual_z"]
    ) / 2.0

# Rank (higher = more anomalous)
rank_col = "diagnostic_anomaly_score_z" if "diagnostic_anomaly_score_z" in anom_df.columns else "input_anomaly_score_z"
anom_df = anom_df.sort_values(rank_col, ascending=False)

# ------------------------------------------------------------
# Output: show top anomalous runs
# ------------------------------------------------------------
print(f"PCs used: {pcs_for_anomaly}")
print(f"Ranking by: {rank_col}")
display(anom_df.head(15))

# ------------------------------------------------------------
# OPTIONAL: export to global_model_outputs (if fp is defined)
# ------------------------------------------------------------
# from pathlib import Path
# out_dir = GLOBAL_OUTPUT_DIR
# out_dir.mkdir(parents=True, exist_ok=True)
# anom_df.to_csv(out_dir / "anomaly_scores.csv", index=False)
# print(f"\nExported: {out_dir / 'anomaly_scores.csv'}")

In [ ]:
# ------------------------------------------------------------
# Elastic Net feature selection (CLASSIFICATION)
#   1) Binary quantile-binning of the target (LOW vs HIGH)
#   2) Logistic regression with Elastic Net penalty (sparse selection)
#   3) Selected features = non-zero coefficients
#
# Assumes you already have:
#   model_df : your prepared modeling dataframe (features + target)
# ------------------------------------------------------------

# ===== USER CONTROLS =====
target_col = TARGET_COL
q = 0.50                 # binary median split (LOW <= median, HIGH > median)
n_splits = 5             # stratified CV folds
random_state = 0
# =========================

# --- Split X / y (numeric target) ---
y_cont = pd.to_numeric(model_df[target_col], errors="coerce")
X = model_df.drop(columns=[target_col]).copy()

# Drop rows with missing target
mask = y_cont.notna()
X, y_cont = X.loc[mask], y_cont.loc[mask]

# --- Binary quantile binning ---
thr = y_cont.quantile(q)
y_bin = (y_cont > thr).astype(int)   # 0 = LOW, 1 = HIGH

print(f"Binary binning threshold at q={q:.2f}: {thr:.4f}")
print("Class counts (0=LOW, 1=HIGH):")
print(y_bin.value_counts().sort_index())

# --- Identify feature types ---
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print(f"\nRows: {X.shape[0]} | Predictors: {X.shape[1]} (numeric={len(num_cols)}, categorical={len(cat_cols)})")

# --- Preprocess: scale numeric, one-hot encode categorical ---
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# --- Elastic Net Logistic Regression with CV ---
# - penalty='elasticnet' requires solver='saga'
# - Cs controls inverse-regularization strength; this grid is reasonable for small N
cv = StratifiedKFold(n_splits=min(n_splits, int(y_bin.value_counts().min())), shuffle=True, random_state=random_state)

logit_enet = LogisticRegressionCV(
    penalty="elasticnet",
    solver="saga",
    l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    Cs=np.logspace(-3, 3, 25),
    cv=cv,
    scoring="roc_auc",
    max_iter=20000,
    n_jobs=-1,
    refit=True
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", logit_enet)
])

pipe.fit(X, y_bin)

# --- Extract selected (non-zero) coefficients ---
feature_names = pipe.named_steps["prep"].get_feature_names_out()
coefs = pipe.named_steps["clf"].coef_.ravel()  # binary -> shape (n_features,)

coef_s = pd.Series(coefs, index=feature_names).sort_values(key=np.abs, ascending=False)

tol = 1e-8
selected = coef_s[coef_s.abs() > tol]

print("\nElastic Net Logistic chosen hyperparameters:")
print(f"  best C:       {pipe.named_steps['clf'].C_[0]}")
print(f"  best l1_ratio: {pipe.named_steps['clf'].l1_ratio_[0]}")

print(f"\nSelected features (non-zero): {selected.shape[0]} / {len(feature_names)}")
display(selected.head(30).to_frame("coef"))

# --- Build selected design matrix (post-preprocess) ---
X_design = pd.DataFrame(pipe.named_steps["prep"].transform(X), columns=feature_names, index=X.index)
X_selected_clf = X_design[selected.index.tolist()].copy()

print("\nShapes for downstream classification modeling:")
print("  X_design:", X_design.shape)
print("  X_selected_clf:", X_selected_clf.shape)

# Optional outputs you’ll likely want later:
# - X_selected_clf : feature-selected matrix for RF classifier
# - y_bin          : binary labels aligned to X_selected_clf.index
# - thr            : the threshold used for binning (save for future inference)

In [ ]:
# Random Forest classification with feature selection refit inside each CV fold.
target_col = TARGET_COL
q = 0.50
n_splits = 5
n_repeats = 20
random_state = 0

y_cont_full = pd.to_numeric(model_df[target_col], errors="coerce")
X_full = model_df.drop(columns=[target_col]).copy()

mask = y_cont_full.notna()
X_full, y_cont_full = X_full.loc[mask], y_cont_full.loc[mask]

thr = y_cont_full.quantile(q)
y_full = (y_cont_full > thr).astype(int)

def fit_classification_selector(X_train, y_train, random_state=0):
    """Fit preprocessing and logistic Elastic Net selection on training rows only."""
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [col for col in X_train.columns if col not in num_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    min_class = int(y_train.value_counts().min())
    cv_inner = StratifiedKFold(
        n_splits=min(n_splits, min_class),
        shuffle=True,
        random_state=random_state,
    )

    logit_enet = LogisticRegressionCV(
        penalty="elasticnet",
        solver="saga",
        l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
        Cs=np.logspace(-3, 3, 25),
        cv=cv_inner,
        scoring="roc_auc",
        max_iter=20000,
        n_jobs=-1,
        refit=True,
    )

    pipe = Pipeline(steps=[("prep", preprocess), ("clf", logit_enet)])
    pipe.fit(X_train, y_train)

    feature_names = pipe.named_steps["prep"].get_feature_names_out()
    coefs = pd.Series(pipe.named_steps["clf"].coef_.ravel(), index=feature_names)
    selected_names = coefs.index[coefs.abs() > 1e-8].tolist()
    if not selected_names:
        selected_names = feature_names.tolist()

    return pipe, selected_names

def transform_selected_clf(selector_pipe, X_data, selected_names):
    feature_names = selector_pipe.named_steps["prep"].get_feature_names_out()
    design = pd.DataFrame(
        selector_pipe.named_steps["prep"].transform(X_data),
        columns=feature_names,
        index=X_data.index,
    )
    return design[selected_names].copy()

roc_auc_scores = []
bal_acc_scores = []
selected_counts = []

for seed in range(n_repeats):
    cv_outer = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    for train_idx, test_idx in cv_outer.split(X_full, y_full):
        X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
        y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]

        selector_pipe, selected_names = fit_classification_selector(X_train, y_train, random_state=seed)
        X_train_sel = transform_selected_clf(selector_pipe, X_train, selected_names)
        X_test_sel = transform_selected_clf(selector_pipe, X_test, selected_names)
        selected_counts.append(len(selected_names))

        rf = RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=seed,
        )

        rf.fit(X_train_sel, y_train)

        y_prob = rf.predict_proba(X_test_sel)[:, 1]
        y_pred = rf.predict(X_test_sel)

        roc_auc_scores.append(roc_auc_score(y_test, y_prob))
        bal_acc_scores.append(balanced_accuracy_score(y_test, y_pred))

roc_auc_scores = np.array(roc_auc_scores)
bal_acc_scores = np.array(bal_acc_scores)

print("Random Forest classification performance with fold-local feature selection:")
print(f"ROC AUC: mean={roc_auc_scores.mean():.3f}, std={roc_auc_scores.std():.3f}")
print("ROC AUC quantiles:", np.quantile(roc_auc_scores, [0.1, 0.25, 0.5, 0.75, 0.9]))
print(f"Balanced Accuracy: mean={bal_acc_scores.mean():.3f}, std={bal_acc_scores.std():.3f}")
print("Balanced Accuracy quantiles:", np.quantile(bal_acc_scores, [0.1, 0.25, 0.5, 0.75, 0.9]))
print(f"Selected feature count: median={np.median(selected_counts):.0f}, range=({np.min(selected_counts)}, {np.max(selected_counts)})")

In [ ]:
rf_clf_final = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=0
)
rf_clf_final.fit(X_selected_clf, y_bin)

In [ ]:
# --- SHAP for your FINAL Random Forest CLASSIFIER (binary) ---
# Assumes you already have:
#   rf_clf_final     -> trained RandomForestClassifier
#   X_selected_clf   -> DataFrame of features used to train rf_clf_final
#   fp               -> original csv filepath (for output folder), optional but recommended

X_explain = X_selected_clf.copy()

explainer = shap.TreeExplainer(rf_clf_final)
shap_vals = explainer.shap_values(X_explain)

# ------------------------------------------------------------
# Normalize SHAP output to a 2D matrix for the positive class (class 1)
# ------------------------------------------------------------
if isinstance(shap_vals, list):
    # Common older behavior: list of [class0, class1]
    shap_pos = shap_vals[1]
else:
    shap_pos = shap_vals

shap_pos = np.asarray(shap_pos)

if shap_pos.ndim == 3:
    # Newer behavior: (n_samples, n_features, n_classes)
    shap_pos = shap_pos[:, :, 1]   # select class 1

if shap_pos.ndim != 2:
    raise ValueError(f"Unexpected SHAP array shape after processing: {shap_pos.shape}")

# ------------------------------------------------------------
# Plots (class 1: HIGH)
# NOTE: use plot_size to avoid narrow/tall issues
# ------------------------------------------------------------
shap.summary_plot(
    shap_pos,
    X_explain,
    max_display=20,
    plot_size=(14, 6)
)

shap.summary_plot(
    shap_pos,
    X_explain,
    plot_type="bar",
    max_display=20,
    plot_size=(14, 6)
)

# ------------------------------------------------------------
# Export SHAP values (class 1) to global_model_outputs
# ------------------------------------------------------------
out_dir = GLOBAL_OUTPUT_DIR if "fp" in globals() else Path.cwd()
out_dir.mkdir(parents=True, exist_ok=True)

shap_df = pd.DataFrame(shap_pos, columns=X_explain.columns, index=X_explain.index)
shap_df.to_csv(out_dir / "shap_values_class1.csv", index=False)

# Optional: mean |SHAP| importance table
imp = shap_df.abs().mean(axis=0).sort_values(ascending=False)
imp.to_csv(out_dir / "shap_importance_class1.csv", header=["mean_abs_shap"])

print(f"Exported:\n  {out_dir / 'shap_values_class1.csv'}\n  {out_dir / 'shap_importance_class1.csv'}")

In [ ]:
# ------------------------------------------------------------
# Contextual anomaly detection (VALID use-cases):
#   A) Input-space anomaly: unusual conditions in PCA space (model-free)
#   B) Contextual anomaly: classifier says HIGH but observed LOW (and vice versa)
#   C) Uncertainty: near-boundary probabilities (diagnostic, not failure prediction)
#
# Requires:
#   score_df          : PCA scores with PC columns
#   y_bin             : binary labels aligned to runs (0=LOW, 1=HIGH)
#   rf_clf_final      : trained RF classifier
#   X_selected_clf    : feature matrix used for classifier (aligned index)
#   df                : original filtered df (optional for identifiers)
#   fp                : original filepath (optional; for export folder)
# ------------------------------------------------------------

# ===== USER CONTROLS =====
pcs_for_context = [f"PC{i}" for i in range(1, 17)]   # PCs used for input anomaly scoring
prob_margin = 0.10    # "near boundary" if |p(HIGH)-0.5| <= margin
random_state = 0
do_cv_probs = True    # True = use out-of-fold probabilities (recommended). False = use in-sample probs.
n_splits = 5          # for out-of-fold probabilities
# =========================

# ---- Align indices ----
idx = X_selected_clf.index.intersection(y_bin.index)
Xc = X_selected_clf.loc[idx].copy()
y = y_bin.loc[idx].copy()

# ---- Predicted probabilities for HIGH (class 1) ----
if do_cv_probs:
    # Out-of-fold probabilities reduce optimism (recommended for diagnostics)
    oof_prob = pd.Series(index=idx, dtype=float)
    cv = StratifiedKFold(n_splits=min(n_splits, int(y.value_counts().min())), shuffle=True, random_state=random_state)
    for tr, te in cv.split(Xc, y):
        rf_tmp = rf_clf_final.__class__(**rf_clf_final.get_params())
        rf_tmp.fit(Xc.iloc[tr], y.iloc[tr])
        oof_prob.iloc[te] = rf_tmp.predict_proba(Xc.iloc[te])[:, 1]
    p_high = oof_prob
else:
    # In-sample probabilities (optimistic; OK for exploration)
    p_high = pd.Series(rf_clf_final.predict_proba(Xc)[:, 1], index=idx)

pred_class = (p_high >= 0.5).astype(int)

# ---- Input-space anomaly in PCA space ----
pcs_for_context = [pc for pc in pcs_for_context if pc in score_df.columns]
if len(pcs_for_context) == 0:
    raise ValueError("No PCs found for contextual anomaly scoring. Check pcs_for_context vs score_df columns.")

S = score_df.loc[idx, pcs_for_context].copy()
S = S.replace([np.inf, -np.inf], np.nan).fillna(S.median(numeric_only=True))

# Robust Mahalanobis distance (structure anomaly)
mcd = MinCovDet(random_state=random_state).fit(S.values)
md2 = mcd.mahalanobis(S.values)
md = np.sqrt(md2)
md_z = (md - md.mean()) / (md.std(ddof=0) + 1e-12)

# Isolation Forest anomaly (structure anomaly)
iso = IsolationForest(n_estimators=500, contamination="auto", random_state=random_state).fit(S.values)
iso_anom = -iso.decision_function(S.values)
iso_z = (iso_anom - iso_anom.mean()) / (iso_anom.std(ddof=0) + 1e-12)

input_anom_z = (md_z + iso_z) / 2.0

# ---- Contextual anomaly signals ----
misclass_flag = (pred_class != y).astype(int)  # 1 = predicted opposite of observed
confidence = (p_high - 0.5).abs()              # distance from decision boundary
uncertainty = 0.5 - confidence                 # higher = more uncertain
near_boundary_flag = (confidence <= prob_margin).astype(int)

# "Strong contradiction" = misclassified AND confident
# (these are the runs that most deserve review, NOT "failures")
strong_contradiction = ((misclass_flag == 1) & (confidence >= prob_margin)).astype(int)

# Combined contextual score (diagnostic ranking only):
# - reward: misclassification (context mismatch)
# - reward: input anomaly (unusual conditions)
# - reward: confidence (if it's confidently wrong, it's more interesting)
context_score = (
    1.0 * misclass_flag +
    0.5 * strong_contradiction +
    0.5 * input_anom_z +
    0.25 * confidence
)

# ---- Assemble report ----
id_cols = [c for c in ["run_id", "run_start_date", "unit_label", "group_label"] if c in df.columns]
report = pd.DataFrame(index=idx)

if id_cols:
    report = pd.concat([df.loc[idx, id_cols], report], axis=1)

report["y_bin_observed"] = y
report["p_high"] = p_high
report["y_bin_pred"] = pred_class

report["misclass_flag"] = misclass_flag
report["near_boundary_flag"] = near_boundary_flag
report["strong_contradiction_flag"] = strong_contradiction

report["mdistance_z"] = md_z
report["iso_anomaly_z"] = iso_z
report["input_anomaly_score_z"] = input_anom_z

report["confidence"] = confidence
report["contextual_anomaly_score"] = context_score

# Sort: highest contextual anomaly first
report = report.sort_values("contextual_anomaly_score", ascending=False)

print(f"Using {'out-of-fold' if do_cv_probs else 'in-sample'} probabilities.")
print(f"Boundary margin: ±{prob_margin:.2f} around 0.5")
display(report.head(15))

# ---- Optional export ----
out_dir = GLOBAL_OUTPUT_DIR if "fp" in globals() else Path.cwd()
out_dir.mkdir(parents=True, exist_ok=True)
report.to_csv(out_dir / "contextual_anomaly_report.csv", index=False)
print(f"\nExported: {out_dir / 'contextual_anomaly_report.csv'}")

In [ ]:
# --- LIME local explanation + visualizations (binary RF classifier) ---

# ---------------------------
# USER CONTROLS
# ---------------------------
run_to_explain = "BH49C"       # <-- set Run ID
label_names = ["LOW", "HIGH"]  # class 0, class 1
num_features = 15
random_state = 0

plot_both_classes = False      # True -> also plot LOW explanation
# ---------------------------

# ---- Checks / alignment ----
if "run_id" not in df.columns:
    raise ValueError("Column 'Run' not found in df.")

X_train = X_selected_clf.copy()
idx = X_train.index

# Find the row index for this Run among rows used by the classifier
run_series = df.loc[idx, "run_id"].astype(str)
matches = run_series.index[run_series == str(run_to_explain)].tolist()
if len(matches) == 0:
    raise ValueError(f"Run {run_to_explain!r} not found among classifier rows (X_selected_clf).")
if len(matches) > 1:
    print(f"Warning: {len(matches)} rows match Run={run_to_explain!r}. Using the first.")

row_idx = matches[0]
x0 = X_train.loc[row_idx].values

# >>> REQUIRED EDIT: get observed class <<<
y_obs = y_bin.loc[row_idx]

# ---- Predict with feature names preserved (avoids sklearn warning) ----
def predict_proba_with_names(X_np):
    X_df = pd.DataFrame(X_np, columns=X_train.columns)
    return rf_clf_final.predict_proba(X_df)

p = predict_proba_with_names(X_train.loc[[row_idx]].values)[0]
pred_label = int(p[1] >= 0.5)

print(f"Run = {run_to_explain!r}")
print(f"Observed class = {label_names[int(y_obs)]}")
print(f"Predicted P(LOW)={p[0]:.3f}, P(HIGH)={p[1]:.3f}  -> predicted={label_names[pred_label]}")

# ---- Build LIME explainer ----
explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=label_names,
    mode="classification",
    discretize_continuous=True,
    random_state=random_state
)

# Force both labels so exp.as_list(label=1) always exists
exp = explainer.explain_instance(
    data_row=x0,
    predict_fn=predict_proba_with_names,
    num_features=min(num_features, X_train.shape[1]),
    labels=(0, 1)
)

# ---- Convert LIME lists to DataFrames for plotting ----
lime_high = pd.DataFrame(exp.as_list(label=1), columns=["feature_rule", "weight"])
lime_low  = pd.DataFrame(exp.as_list(label=0), columns=["feature_rule", "weight"])

# For readability, plot in ascending order so the biggest bars end up on top
lime_high = lime_high.sort_values("weight")
lime_low  = lime_low.sort_values("weight")

# ============================================================
# VISUALIZATION 1: predicted probabilities bar
# ============================================================
plt.figure(figsize=(6, 3))
plt.bar(label_names, [p[0], p[1]])
plt.ylim(0, 1)
plt.ylabel("Predicted probability")
plt.title(f"Run {run_to_explain}: predicted class = {label_names[pred_label]}")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

# ============================================================
# VISUALIZATION 2: LIME contributions toward HIGH
# ============================================================
plt.figure(figsize=(12, 0.45 * len(lime_high) + 1))
plt.barh(lime_high["feature_rule"], lime_high["weight"])
plt.axvline(0, linewidth=1)
plt.xlabel("LIME weight (positive pushes toward HIGH, negative pushes away)")
plt.title(f"LIME explanation toward HIGH (class 1) — Run {run_to_explain}")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# LIME contributions toward LOW (always show)
# ============================================================
plt.figure(figsize=(12, 0.45 * len(lime_low) + 1))
plt.barh(lime_low["feature_rule"], lime_low["weight"])
plt.axvline(0, linewidth=1)
plt.xlabel("LIME weight (positive pushes toward LOW, negative pushes away)")
plt.title(f"LIME explanation toward LOW (class 0) — Run {run_to_explain}")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EWMA drift visualization + drift significance checks
# ============================================================

# ------------- USER CONTROLS -------------
feature_to_plot = "PC4"          # e.g. "PC2"
date_col = "run_start_date"

ewma_span = 7                    # larger = smoother; try 5–15 for small N
band_k = 2.0                     # band width in "EWMA residual SD" units

show_points = True
show_ewma = True
show_band = True

# Drift tests
split_quantile = 0.5             # compare early vs late (median split). try 0.33 for tertiles etc.
min_n_for_tests = 8              # avoid over-interpreting tiny N
# ----------------------------------------

# ---------------------------
# Choose a source dataframe that contains both dates and the feature
# ---------------------------
if "pca_scores_out" in globals() and date_col in pca_scores_out.columns:
    ts_df = pca_scores_out.copy()
elif date_col in df.columns:
    ts_df = df[[date_col]].copy()
    if feature_to_plot in score_df.columns and feature_to_plot not in ts_df.columns:
        ts_df = ts_df.join(score_df[[feature_to_plot]], how="left")
    else:
        if feature_to_plot in df.columns:
            ts_df[feature_to_plot] = df[feature_to_plot]
else:
    raise ValueError(f"Couldn't find '{date_col}' in either pca_scores_out or df.")

if feature_to_plot not in ts_df.columns:
    raise ValueError(
        f"Feature '{feature_to_plot}' not found. Sample available columns: {list(ts_df.columns)[:20]}"
    )

# ---------------------------
# Clean / coerce
# ---------------------------
ts_df = ts_df.copy()
ts_df[date_col] = pd.to_datetime(ts_df[date_col], errors="coerce")
ts_df[feature_to_plot] = pd.to_numeric(ts_df[feature_to_plot], errors="coerce")

ts_df = ts_df.dropna(subset=[date_col, feature_to_plot]).sort_values(date_col)

if ts_df.shape[0] < 3:
    raise ValueError("Not enough non-missing rows to plot drift (need at least 3).")

# ---------------------------
# EWMA + EWMA residual-variance band
# ---------------------------
x = ts_df[feature_to_plot]
ewma = x.ewm(span=ewma_span, adjust=False).mean()

# Approximate local variability: EWMA of squared residuals from EWMA mean
resid = x - ewma
ewma_var = (resid ** 2).ewm(span=ewma_span, adjust=False).mean()
ewma_sd = np.sqrt(ewma_var)

upper = ewma + band_k * ewma_sd
lower = ewma - band_k * ewma_sd

# ---------------------------
# Plot
# ---------------------------
plt.figure(figsize=(12, 5))

if show_points:
    plt.plot(ts_df[date_col], x, marker="o", linestyle="-", alpha=0.7)

if show_band and show_ewma:
    plt.fill_between(ts_df[date_col], lower.values, upper.values, alpha=0.2)

if show_ewma:
    plt.plot(ts_df[date_col], ewma, linewidth=2)

plt.title(f"{feature_to_plot} over time with EWMA (span={ewma_span})")
plt.xlabel("run_start_date")
plt.ylabel(feature_to_plot)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Drift significance checks (directional trend)
# ============================================================
N = len(ts_df)
print(f"N={N} non-missing rows for {feature_to_plot}.")

if N < min_n_for_tests:
    print(f"Note: N < {min_n_for_tests}. Trend tests may be unstable; interpret cautiously.")
else:
    # Convert date to numeric order for rank-based trend tests
    t = ts_df[date_col].view("int64").values  # nanoseconds since epoch (monotonic with time)

    # 1) Spearman correlation: monotonic drift
    rho, p_rho = spearmanr(t, x.values)
    print(f"Spearman(time, {feature_to_plot}): rho={rho:.3f}, p={p_rho:.3g}")

    # 2) Kendall tau: monotonic drift (often more robust at small N)
    tau, p_tau = kendalltau(t, x.values)
    print(f"Kendall(time, {feature_to_plot}):  tau={tau:.3f}, p={p_tau:.3g}")

    # 3) Early vs late shift test (median split by time by default)
    cut = ts_df[date_col].quantile(split_quantile)
    early = x.loc[ts_df[date_col] <= cut]
    late  = x.loc[ts_df[date_col] >  cut]

    if len(early) >= 3 and len(late) >= 3:
        # Mann–Whitney U: non-parametric location shift
        U, p_u = mannwhitneyu(early.values, late.values, alternative="two-sided")
        delta = late.median() - early.median()
        print(f"Early vs late (q={split_quantile:.2f} split at {cut.date()}):")
        print(f"  n_early={len(early)}, n_late={len(late)}")
        print(f"  median(late) - median(early) = {delta:.3g}")
        print(f"  Mann–Whitney p={p_u:.3g}")
    else:
        print("Early/late split too small to test reliably (need >=3 points per group).")

    # Simple “meaningful drift” heuristic summary (customize if you want)
    # - at least one monotonic test significant AND early/late shift test significant
    drift_flag = (p_rho < 0.05 or p_tau < 0.05)
    print(f"\nDirectional drift signal (heuristic): {'YES' if drift_flag else 'NO'} "
          f"(based on monotonic trend tests at p<0.05)")